In [1]:
#Imports and Loading JSON Laws
import json #Imports python's json library
import os #imports the operating system library to read files inside a folder and build file paths
import re #imports regex library
import numpy as np #We need this library for numerical computing

def load_all_laws(folder_path):
    documents = []
    metadata = []

    for file in os.listdir(folder_path):
        if file.endswith(".json"):
            law_name = file.replace(".json", "")
            
            with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:
                data = json.load(f)

                for sec_no, section in data.items():

                    title = section.get("title", "")
                    content = section.get("content", "")

                    doc_text = f"{sec_no} : {title} {content}"

                    documents.append(doc_text)

                    metadata.append({
                        "law": law_name,
                        "section": sec_no,
                        "title": title,
                        "content": content
                    })

    return documents, metadata


# Load Laws
documents, metadata = load_all_laws("laws")

print("Total Documents:", len(documents))

#For checking
#print(document[0])
#print(metadata[0])

Total Documents: 1593


In [2]:
#Preprocessing
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

# Ensure stopwords are available
try:
    stop_words = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords')
    stop_words = set(stopwords.words('english'))

# Manual stopword definition for legal terms
legal_stopwords = {
    "shall", "may", "act", "law", "section", "court", "person",
    "thereof", "therein", "hereby", "upon", "within", "without"
}

stop_words.update(legal_stopwords)

#For Lowercasing and Removing Punctuation
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]','',text)
    return text

#For Tokenization 
def tokenize(text):
    return text.split()

#For removing stopwords
def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

def preprocess(text):
    text = clean_text(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    return tokens

#print(preprocess(documents[0]))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bilak\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
#TF-IDF Model
#Formula = tfidf(t,d) = tf(t,d) × log(N/df(t))

#Preprocess Documents
tokenized_docs = [preprocess(doc) for doc in documents]

# Build vocabulary
vocab = set()
for doc in tokenized_docs:
    vocab.update(doc)

vocab = list(vocab)
vocab_index = {word:i for i, word in enumerate(vocab)}

N = len(tokenized_docs) #Docs no.
V = len(vocab) #Unique tokens no.
tf_matrix = np.zeros((N,V))

for doc_idx, meta in enumerate(metadata):
    
    title = meta["title"]
    content = meta["content"]

    title_tokens = preprocess(title)
    content_tokens = preprocess(content)

    tokens = []

    # Title words weighted higher
    for word in title_tokens:
        tokens.extend([word]*3)

    # Content words normal
    tokens.extend(content_tokens)

    for word in tokens:
        if word in vocab_index:
            tf_matrix[doc_idx][vocab_index[word]] += 1

    if len(tokens) > 0:
        tf_matrix[doc_idx] = tf_matrix[doc_idx] / len(tokens)
        
#Document Frequency
df = np.zeros(V)

for word, idx in vocab_index.items():
    count = 0
    for doc in tokenized_docs:
        if word in doc:
            count += 1
    df[idx] = count

# IDF
idf = np.log(N / (df + 1))

# TF-IDF Matrix
document_matrix = tf_matrix * idf

#To verify indexing
print("TF-IDF Matrix Shape:", document_matrix.shape)

TF-IDF Matrix Shape: (1593, 6674)


In [4]:
#Search Function
#Cosine Similarity Formula: cos(theta) = (A · B) / (||A|| ||B||)

def vectorize_query(query):
    
    tokens = preprocess(query)
    vec = np.zeros(V)

    for word in tokens:
        if word in vocab_index:
            idx = vocab_index[word]
            vec[idx] += 1

    if len(tokens) > 0:
        vec = vec / len(tokens)

    vec = vec * idf

    return vec, tokens


def cosine_similarity(vec1, vec2):
    dot = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)

    if norm1 == 0 or norm2 == 0:
        return 0

    return dot / (norm1 * norm2)


def search(query, top_k=3):

    query_vec, keywords = vectorize_query(query)

    scores = []

    for i, doc_vec in enumerate(document_matrix):
        sim = cosine_similarity(query_vec, doc_vec)
        scores.append((i, sim))

    scores.sort(key=lambda x: x[1], reverse=True)

    top_results = scores[:top_k]

    results = []

    for idx, score in top_results:
        results.append((metadata[idx], score))

    return results, keywords

In [5]:
#Sentence Scoring for Rule Based Summarization

def summarize_section(section, query_keywords):

    text = section["content"]

    sentences = re.split(r'(?<=[.!?]) +', text)

    scored = []

    for sentence in sentences:

        score = 0
        words = preprocess(sentence)

        for word in words:
            if word in query_keywords:
                score += 1

        scored.append((sentence, score))

    scored.sort(key=lambda x: x[1], reverse=True)

    top_sentences = [s[0] for s in scored[:2]]

    return " ".join(top_sentences)

In [6]:
# LLaMA Layer
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Load quantized model for inference
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",          # automatically maps layers to GPU/CPU
    torch_dtype=torch.float16,  # reduces memory usage
    low_cpu_mem_usage=True,
)

# Function to summarize top sections using LLaMA
def llama_summarize(top_sections, query, max_tokens=200):
    
    # Combine context
    context = ""
    for meta, _ in top_sections:
        context += f"{meta['law']} Section {meta['section']}: {meta['title']} - {meta['content']}\n"

    # Prompt for the LLM
    prompt = f"Question: {query}\n\nContext:\n{context}\n\nPlease summarize the answer concisely, citing sections where relevant."

    # Tokenize and move to model device
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # Generate summary
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=False
    )

    # Decode output
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

C:\Users\bilak\Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
C:\Users\bilak\Project\venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bilak\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python

In [20]:
# Full QA Pipeline with Section Summarization (Ready for AI Legal Assistant)
from IPython.display import display, HTML

def summarize_legal_sections(question, top_k=3, max_tokens=150, max_section_chars=500):
    """
    Retrieves top relevant legal sections and produces a concise, easy-to-understand summary.
    
    top_k: number of top sections to retrieve
    max_tokens: max tokens for summarizer output
    max_section_chars: truncate long sections to avoid model overload
    """

    # Step 1: Retrieve top sections using TF-IDF + cosine similarity
    top_sections, keywords = search(question, top_k=top_k)

    print("\nQuestion:", question)
    print("\nTop Relevant Sections:\n")

    truncated_sections = []
    for meta, score in top_sections:
        # Display raw law sections
        print(f"{meta['law']} - Section {meta['section']} : {meta['title']}")
        print(f"Similarity Score: {round(score,3)}")
        print("Answer (raw law text):", meta['content'])
        print("-"*60)

        # Truncate section content for summarizer
        content = meta['content']
        if len(content) > max_section_chars:
            content = content[:max_section_chars] + " [...]"

        truncated_sections.append(({
            'law': meta['law'],
            'section': meta['section'],
            'title': meta['title'],
            'content': content
        }, score))

    # Step 2: Prepare context for summarizer (numbered for clarity)
    context_text = ""
    for i, (sec, _) in enumerate(truncated_sections, 1):
        context_text += f"{i}. {sec['content']}\n"

    # Step 3: Prompt with explicit instructions for concise summary
    prompt = (
        f"{context_text}\n\n"
        "Instructions: Summarize the answer in bullets, only key points, citing section numbers in parentheses. "
        "Use plain English. Do not repeat the full law text."
    )

    # Tokenize and move to model device
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Generate summary
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=False
    )

    # Decode output
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Remove any repeated instructions or prompt artifacts
    summary = summary.replace(prompt, "").strip()

    # Step 4: Clean summary for HTML display
    summary_html = summary.replace("\n", "<br>").replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

    # Step 5: Display summary in collapsible section (raw text) and main answer
    print("\nConcise Legal Summary:\n")
    print(summary)

    html = f"""
    <details>
      <summary><b>Original Law Sections (click to expand)</b></summary>
      <p>
      {"<br><br>".join([f"{sec['law']} - Section {sec['section']} : {sec['title']}<br>{sec['content']}" 
                        for sec, _ in truncated_sections])}
      </p>
    </details>
    """
    display(HTML(html))

In [23]:
summarize_legal_sections("If I hit someone, what legal actions can they take against me?")

Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question: If I hit someone, what legal actions can they take against me?

Top Relevant Sections:

Criminal3 - Section Section 191 : Mutual legal assistance may be effected: If a summons or warrant
Similarity Score: 0.25
Answer (raw law text):  for arrest has to be issued to, or any evidence to be taken from, any offender of a case related to any offence to be proceeded, heard and adjudicated pursuant to this Act who is in a foreign country or it is necessary to assist in the service of a summons or warrant for arrest on, or take any evidence from, any accused of a case sub judice in a foreign court, who is in Nepal, the matter shall be dealt with in accordance with the mutual legal assistance law. 
------------------------------------------------------------
Civil - Section Section 693 : Legal capacity of foreigner to be determined: (1) The legal
Similarity Score: 0.249
Answer (raw law text):  capacity of any foreign natural person shall be determined according to the law of the count